# Building a Singapore Shopping Agent with BuyWhere and OpenAI Function Calling

This cookbook demonstrates how to build a **real-world shopping agent** that uses [BuyWhere](https://buywhere.ai) — a live product catalog API — to answer shopping queries with up-to-date prices across Singapore retailers like Harvey Norman, Shopee, Lazada, and Qoo10.

By the end of this notebook you will know how to:

- Wrap a third-party commerce API as **OpenAI function-calling tools**
- Implement a **multi-turn agent loop** that decides which tools to call and when
- Handle tool errors gracefully inside an agentic workflow
- Run realistic shopping scenarios end-to-end with live pricing data

## Architecture overview

```
User query
    │
    ▼
┌─────────────────────────────────────────┐
│           OpenAI Chat Model             │
│  (decides which BuyWhere tool to call)  │
└────────────┬────────────────────────────┘
             │ tool_calls
             ▼
┌─────────────────────────────────────────┐
│         BuyWhere REST API               │
│  /search  /products/{id}  /deals        │
│  /products/best-price     /categories  │
└────────────┬────────────────────────────┘
             │ tool results
             ▼
┌─────────────────────────────────────────┐
│  Model synthesises a shopping answer    │
└─────────────────────────────────────────┘
```

> **Note:** BuyWhere focuses on the Singapore market (Lazada, Shopee, Harvey Norman, Qoo10) with US retailers (Amazon, Best Buy, Walmart, Target) available in beta.

## Prerequisites

### 1. OpenAI API key
Sign up at [platform.openai.com](https://platform.openai.com) and create an API key at the [API Keys page](https://platform.openai.com/api-keys).

### 2. BuyWhere API key
BuyWhere provides **free API keys** for developers. Generate yours at [buywhere.ai/api-keys](https://buywhere.ai/api-keys).  
The free tier allows **1,000 queries/month** and **10 requests/minute** — more than enough to follow along.

### 3. Create a `.env` file in this directory

```
OPENAI_API_KEY=sk-...
BUYWHERE_API_KEY=bw_live_...
```

## 1. Install dependencies

In [ ]:
%pip install openai requests python-dotenv --quiet

## 2. Imports and configuration

In [ ]:
import os
import json
import requests
from typing import Any
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
BUYWHERE_API_KEY = os.environ["BUYWHERE_API_KEY"]

BUYWHERE_BASE_URL = "https://api.buywhere.io/v1"
BUYWHERE_HEADERS = {"Authorization": f"Bearer {BUYWHERE_API_KEY}"}

MODEL = "gpt-4o"

print("Configuration loaded successfully.")

## 3. BuyWhere API helper functions

We wrap each BuyWhere endpoint in a small Python function. These will be called by the agent when the model issues a tool call.

| Function | Endpoint | Purpose |
|---|---|---|
| `search_products` | `GET /search` | Keyword search across all retailers |
| `get_product_detail` | `GET /products/{id}` | Full details for a single product |
| `get_best_price` | `GET /products/best-price` | Lowest price for a given product query |
| `get_deals` | `GET /deals` | Active promotions and discounts |
| `get_categories` | `GET /categories` | Browse the product taxonomy |

In [ ]:
def _call_buywhere(path: str, params: dict | None = None) -> dict:
    """Make a GET request to the BuyWhere API and return parsed JSON."""
    url = f"{BUYWHERE_BASE_URL}{path}"
    response = requests.get(url, headers=BUYWHERE_HEADERS, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


def search_products(query: str, limit: int = 10) -> dict:
    """
    Search for products by keyword across all supported retailers.

    Args:
        query: Natural-language search term (e.g. '55 inch TV', 'iPhone 15').
        limit: Maximum number of results to return (1–100, default 10).

    Returns:
        JSON response containing a list of matching product records.
    """
    return _call_buywhere("/search", {"q": query, "limit": limit})


def get_product_detail(product_id: str) -> dict:
    """
    Retrieve full details for a specific product including all available offers.

    Args:
        product_id: The BuyWhere product ID returned by a previous search.

    Returns:
        JSON response with complete product information and pricing.
    """
    return _call_buywhere(f"/products/{product_id}")


def get_best_price(query: str) -> dict:
    """
    Find the single lowest-priced offer for a product across all retailers.

    Args:
        query: Product name or description to find the best price for.

    Returns:
        JSON response with the best available price and the retailer offering it.
    """
    return _call_buywhere("/products/best-price", {"q": query})


def get_deals() -> dict:
    """
    Retrieve current deals, promotions, and flash sales from all retailers.

    Returns:
        JSON response with active deals sorted by discount percentage.
    """
    return _call_buywhere("/deals")


def get_categories() -> dict:
    """
    Browse the full product category taxonomy supported by BuyWhere.

    Returns:
        JSON response with all available categories and subcategories.
    """
    return _call_buywhere("/categories")

## 4. Define OpenAI tool schemas

OpenAI function calling requires a JSON Schema description of each tool. The model uses these schemas to decide which tool to call and what arguments to pass.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": (
                "Search for products by keyword across all BuyWhere-supported retailers "
                "(Harvey Norman, Shopee, Lazada, Qoo10, and more). "
                "Use this tool when the user wants to find or compare products."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Natural-language product search term, e.g. '55 inch 4K TV' or 'iPhone 15 Pro 256GB'."
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Number of results to return (1–100). Default is 10.",
                        "default": 10
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_product_detail",
            "description": (
                "Get complete details (specifications, images, all available offers and prices) "
                "for a specific product using its BuyWhere product ID. "
                "Use this after a search to get deeper information about a particular item."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {
                        "type": "string",
                        "description": "The BuyWhere product ID returned from a search_products call."
                    }
                },
                "required": ["product_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_best_price",
            "description": (
                "Find the single lowest price for a product across all retailers. "
                "Use this when the user explicitly asks for the cheapest option."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Product name or description to find the best price for."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_deals",
            "description": (
                "Retrieve today's active deals, flash sales, and promotions across all retailers. "
                "Use this when the user asks about discounts, deals, or what's on sale."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_categories",
            "description": (
                "Browse all available product categories and subcategories in the BuyWhere catalog. "
                "Use this when the user wants to explore what types of products are available."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

# Map tool names to their Python implementations for dispatch
TOOL_DISPATCH = {
    "search_products": search_products,
    "get_product_detail": get_product_detail,
    "get_best_price": get_best_price,
    "get_deals": get_deals,
    "get_categories": get_categories,
}

print(f"Registered {len(TOOLS)} tools: {[t['function']['name'] for t in TOOLS]}")

## 5. The agent loop

The agent loop:
1. Sends the conversation to the model
2. If the model requests tool calls, executes them and appends results
3. Repeats until the model produces a final text response (no more tool calls)

This pattern — sometimes called a **ReAct loop** — is the core of most function-calling agents.

In [ ]:
SYSTEM_PROMPT = """\
You are a helpful Singapore shopping assistant powered by the BuyWhere product catalog API.
You have access to real-time product data from Harvey Norman, Shopee, Lazada, Qoo10, and other retailers.

Guidelines:
- Always use the available tools to fetch live data before answering price-related questions.
- Present prices in Singapore Dollars (SGD) unless the user specifies otherwise.
- When comparing products, highlight key differences in price, specs, and retailer.
- If a tool call fails, inform the user and suggest an alternative query.
- Be concise: lead with the most useful finding, then provide supporting details.
"""


def execute_tool_call(tool_name: str, tool_args: dict) -> str:
    """Dispatch a tool call to its Python implementation and return a JSON string result."""
    if tool_name not in TOOL_DISPATCH:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})
    try:
        result = TOOL_DISPATCH[tool_name](**tool_args)
        return json.dumps(result)
    except requests.HTTPError as e:
        return json.dumps({"error": f"BuyWhere API error {e.response.status_code}: {e.response.text}"})
    except requests.RequestException as e:
        return json.dumps({"error": f"Network error: {str(e)}"})


def run_shopping_agent(user_message: str, conversation_history: list[dict] | None = None) -> tuple[str, list[dict]]:
    """
    Run the shopping agent for a single user turn.

    Args:
        user_message: The user's shopping query.
        conversation_history: Prior messages for multi-turn conversations.
                              Pass None to start a fresh conversation.

    Returns:
        A tuple of (final_answer, updated_conversation_history).
    """
    messages = conversation_history or [{"role": "system", "content": SYSTEM_PROMPT}]
    messages = messages + [{"role": "user", "content": user_message}]

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )

        message = response.choices[0].message
        messages.append(message.model_dump(exclude_unset=True))

        # No tool calls → model produced its final answer
        if not message.tool_calls:
            return message.content, messages

        # Execute every tool call the model requested
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"  → Calling tool: {tool_name}({tool_args})")
            tool_result = execute_tool_call(tool_name, tool_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result,
            })

        # Loop back to let the model process the tool results


print("Agent loop defined successfully.")

## 6. Shopping scenarios

We will walk through five real-world queries that showcase different tools and agent behaviors.

### Scenario 1 — Find the cheapest 55-inch TV in Singapore

The user wants the lowest price. The agent should call `get_best_price` and then potentially `search_products` for context.

In [ ]:
query = "Find me the cheapest 55-inch TV in Singapore"
print(f"User: {query}\n")

answer, history = run_shopping_agent(query)

print(f"\nAssistant: {answer}")

### Scenario 2 — Compare iPhone 15 prices across retailers

The agent should call `search_products` to retrieve multiple listings and present a price comparison table.

In [ ]:
query = "Compare iPhone 15 128GB prices across different Singapore retailers"
print(f"User: {query}\n")

answer, history = run_shopping_agent(query)

print(f"\nAssistant: {answer}")

### Scenario 3 — Browse today's deals

The agent should call `get_deals` and summarise the top promotions currently available.

In [ ]:
query = "What are the best deals available right now?"
print(f"User: {query}\n")

answer, history = run_shopping_agent(query)

print(f"\nAssistant: {answer}")

### Scenario 4 — Gaming laptop under SGD 2,000

A budget-constrained search. The agent searches for products and filters or highlights options within the stated budget.

In [ ]:
query = "Show me gaming laptops available in Singapore under SGD 2000"
print(f"User: {query}\n")

answer, history = run_shopping_agent(query)

print(f"\nAssistant: {answer}")

### Scenario 5 — Multi-turn conversation

We pass the conversation history between turns, allowing the agent to remember context. The user first searches for a product, then asks a follow-up question about a specific result.

In [ ]:
# Turn 1 — initial query
query_1 = "What Sony headphones are available in Singapore?"
print(f"User: {query_1}\n")

answer_1, history = run_shopping_agent(query_1)
print(f"Assistant: {answer_1}\n")
print("-" * 60)

# Turn 2 — follow-up using the same conversation history
query_2 = "Which of those has the best noise cancellation? Get more details on it."
print(f"\nUser: {query_2}\n")

answer_2, history = run_shopping_agent(query_2, conversation_history=history)
print(f"Assistant: {answer_2}")

## 7. Error handling and edge cases

Good agents handle failures gracefully. Let's verify that invalid queries and network errors are caught and reported to the user rather than crashing.

In [ ]:
# Test 1: Valid query — should work normally
print("Test 1: Valid product search")
result = execute_tool_call("search_products", {"query": "Samsung TV", "limit": 3})
parsed = json.loads(result)
if "error" in parsed:
    print(f"  Error: {parsed['error']}")
else:
    print(f"  Returned {len(parsed) if isinstance(parsed, list) else 'data'} results — OK")

print()

# Test 2: Invalid product ID — should return a clean error, not raise an exception
print("Test 2: Non-existent product ID")
result = execute_tool_call("get_product_detail", {"product_id": "invalid-id-xyz-000"})
parsed = json.loads(result)
if "error" in parsed:
    print(f"  Error handled gracefully: {parsed['error']}")
else:
    print("  Returned data")

print()

# Test 3: Unknown tool name — should return a clean error
print("Test 3: Unknown tool")
result = execute_tool_call("nonexistent_tool", {})
parsed = json.loads(result)
print(f"  Error handled gracefully: {parsed['error']}")

## 8. Inspecting the raw API responses

Understanding the shape of BuyWhere API responses helps when customising the agent's prompts or adding new tools.

In [ ]:
# Inspect the structure of a search response
raw_response = search_products("laptop", limit=2)
print("Raw search response structure:")
print(json.dumps(raw_response, indent=2))

In [ ]:
# Inspect available product categories
categories = get_categories()
print("Available categories:")
print(json.dumps(categories, indent=2))

## Conclusion

In this cookbook we built a fully functional Singapore shopping agent by combining:

| Component | What it does |
|---|---|
| **OpenAI function calling** | Lets the model decide which tools to call and with what arguments |
| **BuyWhere REST API** | Provides live product data from Harvey Norman, Shopee, Lazada, Qoo10, and more |
| **Agent loop** | Drives the model ↔ tool cycle until a final answer is ready |
| **Conversation history** | Enables multi-turn follow-ups with full context |
| **Error handling** | Catches API failures and returns clean messages instead of exceptions |

### Next steps

- **Add a price alert tool** — store user preferences and poll BuyWhere for price drops
- **Integrate checkout links** — use BuyWhere's merchant handoff URLs to send users directly to the retailer's cart
- **Extend to the US market** — BuyWhere supports Amazon, Best Buy, Walmart, and Target in beta; swap or add `region=us` to your queries
- **Deploy as a chatbot** — wrap `run_shopping_agent` in a FastAPI endpoint and connect it to a frontend or messaging platform
- **Add streaming** — use `stream=True` with the OpenAI client for a more responsive user experience

### Resources

- [BuyWhere API Documentation](https://buywhere.ai/docs/API_DOCUMENTATION)
- [BuyWhere API Keys](https://buywhere.ai/api-keys)
- [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling)
- [OpenAI Agents SDK](https://github.com/openai/openai-agents-python)